In [1]:
import pandas as pd
import numpy as np
import os
from google.colab import drive
from datetime import timedelta
import re

# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Path to the MIMIC-IV dataset
path = '/content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/patient_data'

Mounted at /content/drive


In [12]:
overalltable_Lab_withventparams = pd.read_csv(f"{path}/overalltable_Lab_withventparams.csv") # Correct 28 non zero columns
overalltable_withoutLab_withventparams2 = pd.read_csv(f"{path}/overalltable_withoutLab_withventparams2.csv")
sampled_lab_withventparams = pd.read_csv(f"{path}/sampled_lab_withventparams.csv")
sampled_withoutlab_withventparams = pd.read_csv(f"{path}/sampled_withoutlab_withventparams.csv")

In [15]:
sampled_withoutlab_withventparams.MechVent.value_counts()

,count
MechVent,


In [14]:
# Find columns in overalltable_Lab_withventparams but not in sampled_lab_withventparams
cols_only_in_overalltable = set(sampled_all_withventparams.columns) - set(sampled_lab_withventparams.columns)

print("Columns in overalltable_Lab_withventparams but not in sampled_lab_withventparams:")
print(cols_only_in_overalltable)

Columns in overalltable_Lab_withventparams but not in sampled_lab_withventparams:
set()


In [ ]:
# Find columns in overalltable_Lab_withventparams but not in sampled_lab_withventparams
cols_only_in_overalltable = set(sampled_lab_withventparams.columns) - set(sampled_withoutlab_withventparams.columns)

print("Columns in overalltable_Lab_withventparams but not in sampled_lab_withventparams:")
print(cols_only_in_overalltable)

Columns in overalltable_Lab_withventparams but not in sampled_lab_withventparams:
{'bicarbonate', 'creatinine', 'pao2', 'ph', 'lactate', 'diasbp', 'albumin', 'bilirubin', 'sodium', 'spo2', 'carbondioxide', 'tempc', 'ptt', 'chloride', 'heartrate', 'inr', 'platelet', 'sgot', 'potassium', 'peep', 'pao2fio2ratio', 'paco2', 'sgpt', 'resprate', 'base_excess', 'meanbp', 'wbc', 'ionizedcalcium', 'sysbp', 'bun', 'bands', 'glucose', 'magnesium', 'hemoglobin', 'calcium', 'pt'}


In [18]:
# Find columns with 100 or more null/NA values in overalltable_Lab_withventparams
null_counts = sampled_all_withventparams.isnull().sum()
cols_with_high_nulls = null_counts[null_counts==2248324]

print("Columns with 100 or more null/NA values in overalltable_Lab_withventparams:")
print(cols_with_high_nulls)

Columns with 100 or more null/NA values in overalltable_Lab_withventparams:
Series([], dtype: int64)


In [ ]:
lab = pd.read_csv(f"{path}/lab_values.csv", nrows=5)
others = pd.read_csv(f"{path}/others_lab_values.csv", nrows=5)
print(lab.head())
print(others.head())

   subject_id     hadm_id   stay_id            charttime  ALBUMIN  ANIONGAP  \
0    10000032  29079034.0  39553978  2180-07-23 21:45:00      NaN      14.0   
1    10000032  29079034.0  39553978  2180-07-24 06:35:00      3.8       9.0   
2    10000690  25860671.0  37081114  2150-11-03 02:56:00      NaN      11.0   
3    10000980  26913865.0  39765666  2189-06-27 20:03:00      NaN       NaN   
4    10000980  26913865.0  39765666  2189-06-28 06:15:00      NaN      17.0   

   BANDS  BICARBONATE  BILIRUBIN  CHLORIDE  ...  SODIUM   BUN  WBC  MAGNESIUM  \
0    NaN         21.0        NaN     102.0  ...   132.0  33.0  NaN        2.3   
1    NaN         24.0        2.7     102.0  ...   130.0  28.0  4.1        2.0   
2    NaN         26.0        NaN     104.0  ...   137.0  21.0  7.5        1.5   
3    NaN          NaN        NaN       NaN  ...     NaN   NaN  NaN        NaN   
4    NaN         23.0        NaN     103.0  ...   139.0  38.0  5.4        2.0   

   CARBONDIOXIDE  BASE_EXCESS  CALCIUM

In [ ]:
overalltable_Lab_withventparams.describe()

,subject_id,hadm_id,stay_id,gcs,HeartRate,SysBP,DiasBP,MeanBP,RespRate,TempC,...,rate_vasopressin,rate_dopamine,vaso_total,iv_total,cum_fluid_balance,PEEP,tidal_volume,plateau_pressure,shockindex,PAO2FiO2ratio
count,8.643150e+05,8.643150e+05,8.643150e+05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
mean,1.499431e+07,2.498876e+07,3.500766e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,2.888183e+06,2.874520e+06,2.884936e+06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,1.000003e+07,2.000009e+07,3.000015e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,1.249485e+07,2.248301e+07,3.252210e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,1.498828e+07,2.500793e+07,3.501621e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,1.750312e+07,2.744788e+07,3.749858e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max,1.999999e+07,2.999983e+07,3.999986e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
import pandas as pd
import numpy as np
import os

def create_overalltable_lab_withventparams(path):
    """
    Recreate SQL logic for overalltable_Lab_withventparams, combining lab_values and others_lab_values tables.

    Parameters:
        path (str): Path to the directory containing input CSVs and where output CSV will be saved.

    Returns:
        pd.DataFrame: Aggregated table as per SQL with averages/maxes per subject_id, hadm_id, stay_id, charttime
    """
    print("Loading input tables...")
    lab = pd.read_csv(f"{path}/lab_values.csv")
    others = pd.read_csv(f"{path}/others_lab_values.csv")

    # Define common identifier columns
    id_cols = ['subject_id', 'hadm_id', 'stay_id', 'charttime']

    # Set columns for union with lab: fill missing 'others' columns with NaN
    lab_cols_fill = {
        'SGOT': np.nan, 'SGPT': np.nan, 'IonizedCalcium': np.nan, # Use correct capitalization
        'MechVent': np.nan, 'FiO2': np.nan, 'urineoutput': np.nan,
        'rate_norepinephrine': np.nan, 'rate_epinephrine': np.nan, 'rate_phenylephrine': np.nan,
        'rate_vasopressin': np.nan, 'rate_dopamine': np.nan, 'vaso_total': np.nan,
        'iv_total': np.nan, 'cum_fluid_balance': np.nan,
        'PEEP': np.nan, 'tidal_volume': np.nan, 'plateau_pressure': np.nan,
        'gcs': np.nan, 'HeartRate': np.nan, 'SysBP': np.nan, 'DiasBP': np.nan,
        'MeanBP': np.nan, 'RespRate': np.nan, 'TempC': np.nan, 'SpO2': np.nan
    }
    for col in lab_cols_fill:
        if col not in lab.columns:
            lab[col] = lab_cols_fill[col]

    # For others: fill missing lab_value cols with NaN and others with NaN similarly
    others_cols_fill = {
        'POTASSIUM': np.nan, 'SODIUM': np.nan, 'CHLORIDE': np.nan, 'GLUCOSE': np.nan,
        'BUN': np.nan, 'CREATININE': np.nan, 'MAGNESIUM': np.nan, 'CALCIUM': np.nan,
        'CARBONDIOXIDE': np.nan, 'BILIRUBIN': np.nan, 'ALBUMIN': np.nan, 'HEMOGLOBIN': np.nan,
        'WBC': np.nan, 'PLATELET': np.nan, 'PTT': np.nan, 'PT': np.nan, 'INR': np.nan, 'PH': np.nan,
        'PAO2': np.nan, 'PACO2': np.nan, 'BASE_EXCESS': np.nan, 'BICARBONATE': np.nan, 'LACTATE': np.nan,
        'BANDS': np.nan, 'MechVent': np.nan, 'FiO2': np.nan, 'urineoutput': np.nan,
        'rate_norepinephrine': np.nan, 'rate_epinephrine': np.nan, 'rate_phenylephrine': np.nan,
        'rate_vasopressin': np.nan, 'rate_dopamine': np.nan, 'vaso_total': np.nan,
        'iv_total': np.nan, 'cum_fluid_balance': np.nan,
        'PEEP': np.nan, 'tidal_volume': np.nan, 'plateau_pressure': np.nan,
        'gcs': np.nan, 'HeartRate': np.nan, 'SysBP': np.nan, 'DiasBP': np.nan,
        'MeanBP': np.nan, 'RespRate': np.nan, 'TempC': np.nan, 'SpO2': np.nan
    }
    for col in others_cols_fill:
        if col not in others.columns:
            others[col] = others_cols_fill[col]

    # Ensure all columns in agg_dict are present in both dataframes before concatenation
    all_cols = list(set(id_cols + list(lab_cols_fill.keys()) + list(others_cols_fill.keys())))
    for col in all_cols:
        if col not in lab.columns:
            lab[col] = np.nan
        if col not in others.columns:
            others[col] = np.nan


    # Concatenate both with same columns (union all) - include identifier columns
    merged = pd.concat([lab[all_cols], others[all_cols]], ignore_index=True)

    # Group by identifiers, aggregating by avg or max per SQL
    agg_dict = {
        'gcs': 'mean', 'HeartRate': 'mean', 'SysBP': 'mean', 'DiasBP': 'mean', 'MeanBP': 'mean',
        'RespRate': 'mean', 'TempC': 'mean', 'SpO2': 'mean',
        'POTASSIUM': 'mean', 'SODIUM': 'mean', 'CHLORIDE': 'mean', 'GLUCOSE': 'mean',
        'BUN': 'mean', 'CREATININE': 'mean', 'MAGNESIUM': 'mean', 'CALCIUM': 'mean',
        'IonizedCalcium': 'mean', # Use correct capitalization
        'CARBONDIOXIDE': 'mean', 'SGOT': 'mean', 'SGPT': 'mean',
        'BILIRUBIN': 'mean', 'ALBUMIN': 'mean', 'HEMOGLOBIN': 'mean', 'WBC': 'mean',
        'PLATELET': 'mean', 'PTT': 'mean', 'PT': 'mean', 'INR': 'mean', 'PH': 'mean',
        'PAO2': 'mean', 'PACO2': 'mean', 'BASE_EXCESS': 'mean', 'BICARBONATE': 'mean',
        'LACTATE': 'mean', 'BANDS': 'mean', 'MechVent': 'mean', 'FiO2': 'mean', # Use correct capitalization
        'urineoutput': 'mean', 'rate_norepinephrine': 'mean', 'rate_epinephrine': 'mean',
        'rate_phenylephrine': 'mean', 'rate_vasopressin': 'mean', 'rate_dopamine': 'mean',
        'vaso_total': 'mean', 'iv_total': 'mean', 'cum_fluid_balance': 'mean',
        'PEEP': 'max', 'tidal_volume': 'max', 'plateau_pressure': 'max'
    }


    # Grouping by identifiers with aggregation
    grouped = merged.groupby(id_cols, as_index=False).agg(agg_dict)

    # Compute shockindex = avg(SysBP) / avg(HeartRate), handle zero division
    grouped['shockindex'] = grouped['SysBP'] / grouped['HeartRate'].replace(0, pd.NA)

    # Compute PaO2FiO2ratio = avg(PaO2)/avg(FiO2)*100, handle zero FiO2
    grouped['PAO2FiO2ratio'] = grouped['PAO2'] / grouped['FiO2'].replace(0, pd.NA) * 100 # Use correct capitalization

    # Coerce MechVent to integer (>=0.5 map to 1)
    grouped['MechVent'] = (grouped['MechVent'] > 0).astype(int)

    # Sort by subject_id, hadm_id, stay_id, charttime
    grouped = grouped.sort_values(id_cols).reset_index(drop=True)

    # Save to CSV
    output_filepath = os.path.join(path, 'overalltable_Lab_withventparams.csv')
    grouped.to_csv(output_filepath, index=False)
    print("Aggregated overall lab with ventilation parameters saved to:", output_filepath)

    return grouped

In [ ]:
overalltable_lab_withventparams = create_overalltable_lab_withventparams(path)
display(overalltable_lab_withventparams.head())

Loading input tables...
Aggregated overall lab with ventilation parameters saved to: /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/patient_data/overalltable_Lab_withventparams.csv


,subject_id,hadm_id,stay_id,charttime,gcs,HeartRate,SysBP,DiasBP,MeanBP,RespRate,...,rate_vasopressin,rate_dopamine,vaso_total,iv_total,cum_fluid_balance,PEEP,tidal_volume,plateau_pressure,shockindex,PAO2FiO2ratio
0,10000032,29079034.0,39553978,2180-07-23 21:45:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10000032,29079034.0,39553978,2180-07-24 06:35:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10000690,25860671.0,37081114,2150-11-03 02:56:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10000980,26913865.0,39765666,2189-06-27 20:03:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10000980,26913865.0,39765666,2189-06-28 06:15:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
overalltable_lab_withventparams.shape

(864315, 55)

In [ ]:
import pandas as pd
import numpy as np

def create_overalltable_withoutLab_withventparams(path):
    """
    Combine multiple CSV data sources following the union-all SQL query logic,
    then aggregate per subject_id, hadm_id, stay_id, charttime with means and maxes.

    Parameters:
        vitalsigns_path (str): CSV path for getVitalsigns2 (vital signs)
        ventilation_params_path (str): CSV path for getVentilationparams2 (vent params)
        urineoutput_path (str): CSV path for getUrineoutput2
        vasopressors_path (str): CSV path for getVasopressors2
        intravenous_path (str): CSV path for getIntravenous2 (IV fluids)
        cumfluid_path (str): CSV path for getCumFluid
        ventparameters_path (str): CSV path for ventparameters (vent params max values)
        output_path (str): CSV path to save the final aggregated table

    Returns:
        pd.DataFrame: aggregated table matching SQL materialized view
    """
    print("Loading source CSV files...")
    # Load each table with required columns, add missing cols for union
    vitals = pd.read_csv(f"{path}/vital_signs2.csv")
    print(vitals.shape)
    vent = pd.read_csv(f"{path}/ventilation_params.csv")
    vent.rename(columns={'fio2_chartevents': 'FiO2'}, inplace=True)
    lab = pd.read_csv(f"{path}/lab_values.csv")
    urine = pd.read_csv(f"{path}/urine_output.csv")
    urine.rename(columns={'urineOutput': 'urineoutput'}, inplace=True)
    vaso = pd.read_csv(f"{path}/vasopressors_combined_final.csv")
    intravenous = pd.read_csv(f"{path}/intravenous.csv")
    intravenous.rename(columns={'amount': 'iv_total'}, inplace=True)
    cumfluid = pd.read_csv(f"{path}/cum_fluid.csv")
    cumfluid.rename(columns={'cum_fluid_balance': 'cum_fluid_balance'}, inplace=True)
    ventparams = pd.read_csv(f"{path}/vent_parameters.csv")
    print(ventparams.shape)

    # Define full column list and initialize dict for filling missing columns per source
    full_columns = [
        'subject_id', 'hadm_id', 'stay_id', 'charttime',
        'gcs', 'HeartRate', 'SysBP', 'DiasBP', 'MeanBP', 'RespRate', 'TempC', 'SpO2',
        'POTASSIUM', 'SODIUM', 'CHLORIDE', 'GLUCOSE', 'BUN', 'CREATININE', 'MAGNESIUM', 'IonizedCalcium',
        'CALCIUM', 'CARBONDIOXIDE',
        'SGOT', 'SGPT', 'BILIRUBIN', 'ALBUMIN', 'HEMOGLOBIN', 'WBC', 'PLATELET', 'PTT', 'PT', 'INR',
        'PH', 'PAO2', 'PACO2',
        'BASE_EXCESS', 'BICARBONATE', 'LACTATE', 'BANDS',
        'MechVent', 'FiO2', 'urineoutput',
        'rate_norepinephrine', 'rate_epinephrine', 'rate_phenylephrine', 'rate_vasopressin', 'rate_dopamine', 'vaso_total',
        'iv_total', 'cum_fluid_balance',
        'PEEP', 'tidal_volume', 'plateau_pressure'
    ]

    def align_columns(df, source_cols):
        """Add missing columns with NaN to source df to match union columns"""
        for col in full_columns:
            if col not in source_cols:
                df[col] = np.nan
        return df[full_columns]

    # Prepare each dataframe with correct columns and renaming
    vitals_aligned = align_columns(vitals, vitals.columns)
    vent_aligned = align_columns(vent, vent.columns)
    lab_aligned = align_columns(lab, lab.columns)
    urine_aligned = align_columns(urine, urine.columns)
    vaso_aligned = align_columns(vaso, vaso.columns)
    intravenous_aligned = align_columns(intravenous, intravenous.columns)
    cumfluid_aligned = align_columns(cumfluid, cumfluid.columns)
    ventparams_aligned = align_columns(ventparams, ventparams.columns)

    # Rename columns in vitals to match case-sensitive names from SQL
    vitals_aligned = vitals_aligned.rename(columns={
        'heartrate': 'HeartRate',
        'sysbp': 'SysBP',
        'diasbp': 'DiasBP',
        'meanbp': 'MeanBP',
        'resprate': 'RespRate',
        'tempc': 'TempC',
        'spo2': 'SpO2'
    })

    # Union all dataframes (concatenate)
    merged = pd.concat([
        vitals_aligned,
        vent_aligned,
        lab_aligned,
        urine_aligned,
        vaso_aligned,
        intravenous_aligned,
        cumfluid_aligned,
        ventparams_aligned
    ], ignore_index=True)
    print(merged.shape)

    # Define aggregation methods per column as per SQL
    agg_funcs = {
        'gcs': 'mean', 'HeartRate': 'mean', 'SysBP': 'mean', 'DiasBP': 'mean', 'MeanBP': 'mean',
        'RespRate': 'mean', 'TempC': 'mean', 'SpO2': 'mean',
        'POTASSIUM': 'mean', 'SODIUM': 'mean', 'CHLORIDE': 'mean', 'GLUCOSE': 'mean',
        'BUN': 'mean', 'CREATININE': 'mean', 'MAGNESIUM': 'mean', 'IonizedCalcium': 'mean', 'CALCIUM': 'mean', 'CARBONDIOXIDE': 'mean',
        'SGOT': 'mean', 'SGPT': 'mean', 'BILIRUBIN': 'mean', 'ALBUMIN': 'mean', 'HEMOGLOBIN': 'mean', 'WBC': 'mean',
        'PLATELET': 'mean', 'PTT': 'mean', 'PT': 'mean', 'INR': 'mean', 'PH': 'mean', 'PAO2': 'mean', 'PACO2': 'mean',
        'BASE_EXCESS': 'mean', 'BICARBONATE': 'mean', 'LACTATE': 'mean', 'BANDS': 'mean',
        'MechVent': 'mean', 'FiO2': 'mean',
        'urineoutput': 'mean',
        'rate_norepinephrine': 'mean', 'rate_epinephrine': 'mean', 'rate_phenylephrine': 'mean', 'rate_vasopressin': 'mean',
        'rate_dopamine': 'mean', 'vaso_total': 'mean',
        'iv_total': 'mean', 'cum_fluid_balance': 'mean',
        'PEEP': 'max', 'tidal_volume': 'max', 'plateau_pressure': 'max'
    }

    # Aggregate by identifier columns
    grouped = merged.groupby(['subject_id', 'hadm_id', 'stay_id', 'charttime'], as_index=False).agg(agg_funcs)

    # Calculate Shock Index: avg(SysBP) / avg(HeartRate)
    grouped['shockindex'] = grouped['SysBP'] / grouped['HeartRate'].replace(0, np.nan)

    # Calculate PaO2/FiO2 ratio, multiply by 100 as per SQL notes (handle zero FiO2)
    grouped['PAO2FiO2ratio'] = grouped['PAO2'] / grouped['FiO2'].replace(0, np.nan) * 100

    # Convert MechVent to integer flag (1 if avg > 0 else 0)
    grouped['MechVent'] = (grouped['MechVent'] > 0).astype(int)

    # Sort final dataframe
    grouped = grouped.sort_values(by=['subject_id', 'hadm_id', 'stay_id', 'charttime']).reset_index(drop=True)

    # Save to CSV
    grouped.to_csv(f"{path}/overalltable_withoutLab_withventparams2.csv", index=False)
    print(f"Saved overalltable_withoutLab_withventparams2 to {path}")

    return grouped


In [ ]:
overalltable_withoutLab_withventparams = create_overalltable_withoutLab_withventparams(path)
display(overalltable_withoutLab_withventparams.head())
print(overalltable_withoutLab_withventparams.describe())

Loading source CSV files...
(12716194, 12)
(979045, 7)
(24164150, 53)
Saved overalltable_withoutLab_withventparams2 to /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/patient_data


,subject_id,hadm_id,stay_id,charttime,gcs,HeartRate,SysBP,DiasBP,MeanBP,RespRate,...,rate_vasopressin,rate_dopamine,vaso_total,iv_total,cum_fluid_balance,PEEP,tidal_volume,plateau_pressure,shockindex,PAO2FiO2ratio
0,10000032.0,29079034.0,39553978,2180-07-23 14:00:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10000032.0,29079034.0,39553978,2180-07-23 14:11:00,NaN,NaN,84.0,48.0,56.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10000032.0,29079034.0,39553978,2180-07-23 14:12:00,NaN,91.0,NaN,NaN,NaN,24.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10000032.0,29079034.0,39553978,2180-07-23 14:13:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10000032.0,29079034.0,39553978,2180-07-23 14:30:00,NaN,93.0,95.0,59.0,67.0,21.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.021505,NaN


         subject_id       hadm_id       stay_id           gcs     HeartRate  \
count  2.057776e+07  2.057776e+07  2.057776e+07  1.687696e+06  8.748423e+06   
mean   1.501387e+07  2.500038e+07  3.497624e+07  1.168982e+01  8.621165e+01   
std    2.893244e+06  2.869958e+06  2.886881e+06  3.782668e+00  1.855423e+01   
min    1.000003e+07  2.000009e+07  3.000015e+07  3.000000e+00  1.000000e+00   
25%    1.250948e+07  2.251676e+07  3.247525e+07  9.000000e+00  7.300000e+01   
50%    1.502895e+07  2.503104e+07  3.497193e+07  1.400000e+01  8.500000e+01   
75%    1.752510e+07  2.745021e+07  3.746813e+07  1.500000e+01  9.800000e+01   
max    1.999999e+07  2.999983e+07  3.999986e+07  1.500000e+01  2.960000e+02   

              SysBP        DiasBP        MeanBP      RespRate         TempC  \
count  8.311770e+06  8.310005e+06  8.603041e+06  8.729615e+06  2.444652e+06   
mean   1.194796e+02  6.317902e+01  7.915784e+01  2.020642e+01  3.696147e+01   
std    2.253658e+01  1.529245e+01  1.627758e+01  5.

In [ ]:
overalltable_withoutLab_withventparams.shape

(20577760, 55)

In [ ]:
'CHLORIDE', 'RespRate', 'IonizedCalcium', 'BASE_EXCESS', 'MeanBP', 'TempC', 'SysBP', 'MAGNESIUM', 'PTT', 'HEMOGLOBIN', 'DiasBP', 'PAO2', 'CARBONDIOXIDE', 'PH', 'INR', 'HeartRate', 'BICARBONATE', 'PLATELET', 'SpO2', 'PAO2FiO2ratio', 'PACO2', 'BUN', 'CREATININE', 'ALBUMIN', 'PT', 'SODIUM', 'GLUCOSE', 'SGOT', 'charttime', 'WBC', 'BANDS', 'PEEP', 'SGPT', 'BILIRUBIN', 'LACTATE', 'CALCIUM', 'POTASSIUM'

In [ ]:
import pandas as pd
import numpy as np

def create_sampled_lab_withventparams(path):
    """
    Mimics the SQL time-binning sample logic, binning ICU encounter data into 4-hour intervals and aggregating.

    Parameters:
        overalltable_path (str): Path to overalltable_lab_withventparams CSV.
        output_path (str): Output CSV path for the sampled (binned) data.

    Returns:
        pd.DataFrame: Sampled and aggregated table as per SQL.
    """
    print("Loading overalltable_lab_withventparams...")
    df = pd.read_csv(f"{path}/overalltable_Lab_withventparams.csv",
                     parse_dates=["charttime"])

    # Step 1: Find min/max charttime for each (subject, hadm, stay) group
    minmax = (
        df.groupby(['subject_id', 'hadm_id', 'stay_id'])['charttime']
        .agg(['min', 'max'])
        .reset_index()
        .rename(columns={'min':'mint', 'max':'maxt'})
    )

    # Step 2: For each group, build grid of 4-hour bins
    print("Generating 4-hour time bins for all stays...")
    grid_records = []
    for idx, row in minmax.iterrows():
        grid_times = pd.date_range(start=row['mint'], end=row['maxt'], freq="4H")
        for t in grid_times:
            grid_records.append({
                'subject_id': row['subject_id'],
                'hadm_id': row['hadm_id'],
                'stay_id': row['stay_id'],
                'start_time': t
            })
    grid = pd.DataFrame(grid_records)

    # Step 3: Merge as left join where charttime is within [start_time, start_time+4H)
    print("Merging grid with original data for 4-hour bins (this may take a while)...")
    df['key'] = (
        df['subject_id'].astype(str)
        + '_' + df['hadm_id'].astype(str)
        + '_' + df['stay_id'].astype(str)
    )
    grid['key'] = (
        grid['subject_id'].astype(str)
        + '_' + grid['hadm_id'].astype(str)
        + '_' + grid['stay_id'].astype(str)
    )

    df = df.sort_values('charttime')
    grid = grid.sort_values('start_time')

    # Only columns that exist in original table and are needed for aggregation
    columns_needed = [
    'ALBUMIN',  'BANDS',        'BASE_EXCESS',  'BICARBONATE',  'BILIRUBIN',    'BUN',  'CALCIUM',
    'CARBONDIOXIDE',    'CHLORIDE',     'CREATININE',   'cum_fluid_balance',    'DiasBP',       'FiO2', 'gcs',
    'GLUCOSE',  'HeartRate',    'HEMOGLOBIN',   'INR',  'IonizedCalcium',       'iv_total',     'LACTATE',      'MAGNESIUM',
    'MeanBP',   'MechVent',     'PACO2',        'PAO2', 'PAO2FiO2ratio',        'PEEP', 'PH',   'plateau_pressure',     'PLATELET',
    'POTASSIUM',        'PT',   'PTT',  'rate_dopamine',        'rate_epinephrine',     'rate_norepinephrine',
    'rate_phenylephrine',       'rate_vasopressin',     'RespRate',     'SGOT', 'SGPT', 'shockindex',   'SODIUM',
    'SpO2',     'SysBP',        'TempC',        'tidal_volume', 'urineoutput',  'vaso_total',   'WBC'

    ]
    columns_present = [col for col in columns_needed if col in df.columns]


    print("Binning each time window; this can be memory-intensive for very large files.")
    sampled = []
    for name, g in grid.groupby('key'):
        subj, hadm, icu = g['subject_id'].iloc[0], g['hadm_id'].iloc[0], g['stay_id'].iloc[0]
        subj_df = df[df['key'] == name]
        for i, row in g.iterrows():
            start = row['start_time']
            end = start + pd.Timedelta(hours=4)
            in_bin = subj_df[
                (subj_df['charttime'] >= start) &
                (subj_df['charttime'] < end)
            ]
            if not in_bin.empty:
                agg = dict(
                    subject_id=subj, hadm_id=hadm, stay_id=icu, start_time=start
                )
                # Aggregates: mean, sum, max, int flag, per SQL
                # (round gcs, average vital/lab, int for MechVent, sum for urineout/iv_total, max for rates/PEEP/tv/ppress)
                agg['gcs'] = np.round(in_bin['gcs'].mean()) if 'gcs' in columns_present else np.nan
                for col in ['ALBUMIN',  'BANDS',        'BASE_EXCESS',  'BICARBONATE',  'BILIRUBIN',    'BUN',
                            'CALCIUM',  'CARBONDIOXIDE',        'CHLORIDE',     'CREATININE',   'cum_fluid_balance',
                            'DiasBP',   'GLUCOSE',      'HeartRate',    'HEMOGLOBIN',   'INR',  'IonizedCalcium',       'LACTATE',
                            'MAGNESIUM',        'MeanBP',       'PACO2',        'PAO2', 'PAO2FiO2ratio',        'PH',   'PLATELET',
                            'POTASSIUM',        'PT',   'PTT',  'RespRate',     'SGOT', 'SGPT', 'shockindex',   'SODIUM',
                            'SpO2',     'SysBP',        'TempC',        'WBC'
                   ]:
                    if col in columns_present:
                        # Removed .lower() here to keep original capitalization
                        agg[col] = in_bin[col].mean()
                agg['MechVent'] = int((in_bin['MechVent'].mean() if 'MechVent' in columns_present else 0) > 0)
                agg['FiO2'] = in_bin['FiO2'].mean() if 'FiO2' in columns_present else np.nan
                agg['urineoutput'] = in_bin['urineoutput'].sum() if 'urineoutput' in columns_present else np.nan
                agg['iv_total'] = in_bin['iv_total'].sum() if 'iv_total' in columns_present else np.nan
                for col in [
                    'rate_norepinephrine', 'rate_epinephrine', 'rate_phenylephrine', 'rate_vasopressin',
                    'rate_dopamine', 'vaso_total', 'PEEP', 'tidal_volume', 'plateau_pressure'
                ]:
                    if col in columns_present:
                        # Removed .lower() here to keep original capitalization
                        agg[col] = in_bin[col].max()
                sampled.append(agg)
    sampled_df = pd.DataFrame(sampled)

    # Restore key columns as integers where possible
    for c in ['subject_id','hadm_id','stay_id']:
        if c in sampled_df.columns:
            sampled_df[c] = sampled_df[c].astype("Int64")

    # Order by identifiers
    sampled_df = sampled_df.sort_values(['stay_id', 'subject_id', 'hadm_id', 'start_time']).reset_index(drop=True)
    sampled_df.to_csv(f"{path}/sampled_lab_withventparams.csv", index=False)
    print("Saved 4-hour sampled binned file to", path)
    return sampled_df

In [ ]:
sampled_lab_withventparams = create_sampled_lab_withventparams(path)
display(sampled_lab_withventparams.head())

Loading overalltable_lab_withventparams...
Generating 4-hour time bins for all stays...


/tmp/ipython-input-2517052015.py:31: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  grid_times = pd.date_range(start=row['mint'], end=row['maxt'], freq="4H")


Merging grid with original data for 4-hour bins (this may take a while)...
Binning each time window; this can be memory-intensive for very large files.
Saved 4-hour sampled binned file to /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/patient_data


,subject_id,hadm_id,stay_id,start_time,gcs,ALBUMIN,BANDS,BASE_EXCESS,BICARBONATE,BILIRUBIN,...,iv_total,rate_norepinephrine,rate_epinephrine,rate_phenylephrine,rate_vasopressin,rate_dopamine,vaso_total,PEEP,tidal_volume,plateau_pressure
0,12466550,23998182,30000153,2174-09-29 12:27:00,NaN,NaN,NaN,-3.333333,19.0,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,12466550,23998182,30000153,2174-09-29 16:27:00,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,12466550,23998182,30000153,2174-09-30 00:27:00,NaN,NaN,NaN,NaN,23.0,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,12466550,23998182,30000153,2174-09-30 16:27:00,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,13180007,27543152,30000213,2162-06-21 08:27:00,NaN,NaN,NaN,0.000000,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# import pandas as pd
# import numpy as np

# def create_sampled_withoutlab_withventparams_fast(path):
#     """
#     Optimized version using pandas vectorized resampling.
#     Replaces manual looping with groupby().resample('4h').
#     """
#     print("Loading overalltable_withoutlab_withventparams...")
#     df = pd.read_csv(f"{path}/overalltable_withoutLab_withventparams2.csv",
#                      parse_dates=["charttime"])

#     # Ensure datetime index for resampling
#     df = df.sort_values(["stay_id", "charttime"]).set_index("charttime")

#     # Define columns requiring different aggregations
#     mean_cols = [
#         'ALBUMIN',	'BANDS',	'BASE_EXCESS',	'BICARBONATE',	'BILIRUBIN',	'BUN',	'CALCIUM',	'CARBONDIOXIDE',
#         'CHLORIDE',	'CREATININE',	'cum_fluid_balance',	'DiasBP',	'gcs',	'GLUCOSE',	'HeartRate',	'HEMOGLOBIN',
#         'INR',	'IonizedCalcium',	'LACTATE',	'MAGNESIUM',	'MeanBP',	'PACO2',	'PAO2',	'PAO2FiO2ratio',
#         'PH',	'PLATELET',	'POTASSIUM',	'PT',	'PTT',	'RespRate',	'SGOT',	'SGPT',	'shockindex',	'SODIUM',
#         'SpO2',	'SysBP',	'TempC',	'WBC', 'FiO2'
#     ]
#     sum_cols = ["urineoutput","iv_total"]
#     max_cols = [
#         "rate_norepinephrine","rate_epinephrine","rate_phenylephrine",
#         "rate_vasopressin","rate_dopamine","vaso_total","PEEP","tidal_volume","plateau_pressure"
#     ]

#     # Subset columns for aggregation
#     agg_map = {c:"mean" for c in mean_cols}
#     agg_map.update({c:"sum" for c in sum_cols})
#     agg_map.update({c:"max" for c in max_cols})

#     # Efficient group + resample
#     print("Performing vectorized 4-hour resampling...")
#     grouped = (
#         df.groupby(["stay_id", "subject_id", "hadm_id"])
#         .resample("4h")          # bin within each ICU stay
#         .agg(agg_map)
#         .reset_index()
#     )

#     # Recalculate MechVent and round GCS per SQL logic
#     grouped["MechVent"] = (grouped["MechVent"] > 0).astype(int) if "MechVent" in grouped else np.nan
#     grouped["gcs"] = grouped["gcs"].round(0) if "gcs" in grouped else np.nan

#     # Compute derived metrics
#     if "SysBP" in grouped and "HeartRate" in grouped:
#         grouped["shockindex"] = grouped["SysBP"] / grouped["HeartRate"].replace(0,np.nan)
#     if "PaO2" in grouped and "FiO2" in grouped:
#         grouped["PaO2FiO2ratio"] = grouped["PaO2"] / grouped["FiO2"].replace(0,np.nan) * 100

#     # Rename time column to match SQL
#     grouped = grouped.rename(columns={"charttime":"start_time"})
#     # grouped = grouped.sort_values(["icustay_id","start_time"]).reset_index(drop=True)

#     # grouped.to_csv(output_path, index=False)
#     sampled_df = grouped.sort_values(['stay_id','subject_id','hadm_id','start_time']).reset_index(drop=True)
#     sampled_df.to_csv(f"{path}/sampled_withoutlab_withventparams.csv", index=False)
#     print("Saved samples withoutlab_withventparams to", path)
#     return grouped


In [17]:
overalltable_withoutLab_withventparams2 = pd.read_csv(f"{path}/overalltable_withoutLab_withventparams2.csv")
overalltable_withoutLab_withventparams2.MechVent.value_counts()

,count
MechVent,
0,19524280
1,1053480


In [24]:
import pandas as pd
import numpy as np

def create_sampled_withoutlab_withventparams_corrected(path):
    """
    Optimized version using pandas vectorized resampling, corrected to include all SQL logic.
    Replaces manual looping with groupby().resample('4h').
    """
    print("Loading overalltable_withoutlab_withventparams...")
    df = pd.read_csv(f"{path}/overalltable_withoutLab_withventparams2.csv",
                     parse_dates=["charttime"])

    # Ensure datetime index for resampling
    df = df.sort_values(["stay_id", "charttime"]).set_index("charttime")

    # Define columns requiring different aggregations
    mean_cols = [
        'ALBUMIN',      'BANDS',        'BASE_EXCESS',  'BICARBONATE',  'BILIRUBIN',    'BUN',  'CALCIUM',      'CARBONDIOXIDE',
        'CHLORIDE',     'CREATININE',   'DiasBP',       'gcs',  'GLUCOSE',      'HeartRate',    'HEMOGLOBIN',
        'INR',  'IonizedCalcium',       'LACTATE',      'MAGNESIUM',    'MeanBP',       'PACO2',        'PAO2', 'PAO2FiO2ratio',
        'PH',   'PLATELET',     'POTASSIUM',    'PT',   'PTT',  'RespRate',     'SGOT', 'SGPT', 'shockindex',   'SODIUM',
        'SpO2', 'SysBP',        'TempC',        'WBC', 'FiO2', 'cum_fluid_balance', 'MechVent'
    ]
    sum_cols = ["urineoutput","iv_total"]
    max_cols = [
        "rate_norepinephrine","rate_epinephrine","rate_phenylephrine",
        "rate_vasopressin","rate_dopamine","vaso_total","PEEP","tidal_volume","plateau_pressure"
    ]

    # Subset columns for aggregation
    agg_map = {c:"mean" for c in mean_cols if c in df.columns}
    agg_map.update({c:"sum" for c in sum_cols if c in df.columns})
    agg_map.update({c:"max" for c in max_cols if c in df.columns})
    # Ensure MechVent is included if present, aggregated by mean

    print("Performing vectorized 4-hour resampling...")
    grouped = (
        df.groupby(["stay_id", "subject_id", "hadm_id"])
        .resample("4h")          # bin within each ICU stay
        .agg(agg_map)
        .reset_index()
    )

    # Recalculate MechVent and round GCS per SQL logic
    if "MechVent" in grouped.columns: # Added check here
        grouped["MechVent"] = (grouped["MechVent"] > 0).astype(int)
    grouped["gcs"] = grouped["gcs"].round(0) if "gcs" in grouped else np.nan

    # Compute derived metrics
    if "SysBP" in grouped and "HeartRate" in grouped:
        grouped["shockindex"] = grouped["SysBP"] / grouped["HeartRate"].replace(0,np.nan)
    if "PaO2" in grouped and "FiO2" in grouped:
        grouped["PAO2FiO2ratio"] = grouped["PAO2"] / grouped["FiO2"].replace(0,np.nan) * 100


    # Rename time column to match SQL
    grouped = grouped.rename(columns={"charttime":"start_time"})
    # grouped = grouped.sort_values(["icustay_id","start_time"]).reset_index(drop=True)

    # grouped.to_csv(output_path, index=False)
    sampled_df = grouped.sort_values(['stay_id','subject_id','hadm_id','start_time']).reset_index(drop=True)
    sampled_df.to_csv(f"{path}/sampled_withoutlab_withventparams.csv")
    print("Saved samples withoutlab_withventparams to", path)
    return grouped

In [25]:
sampled_withoutlab_withventparams_corrected = create_sampled_withoutlab_withventparams_corrected(path)
display(sampled_withoutlab_withventparams_corrected.head(2))

Loading overalltable_withoutlab_withventparams...
Performing vectorized 4-hour resampling...
Saved samples withoutlab_withventparams to /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/patient_data


,stay_id,subject_id,hadm_id,start_time,ALBUMIN,BANDS,BASE_EXCESS,BICARBONATE,BILIRUBIN,BUN,...,iv_total,rate_norepinephrine,rate_epinephrine,rate_phenylephrine,rate_vasopressin,rate_dopamine,vaso_total,PEEP,tidal_volume,plateau_pressure
0,30000153,12466550.0,23998182.0,2174-09-29 12:00:00,NaN,NaN,-3.0,19.0,NaN,22.0,...,1400.0,NaN,NaN,NaN,NaN,NaN,NaN,5.0,537.5,16.0
1,30000153,12466550.0,23998182.0,2174-09-29 16:00:00,NaN,NaN,-4.0,NaN,NaN,NaN,...,50.0,NaN,NaN,NaN,NaN,NaN,NaN,5.0,626.0,NaN


In [26]:
display(sampled_withoutlab_withventparams_corrected.describe())

,stay_id,subject_id,hadm_id,start_time,ALBUMIN,BANDS,BASE_EXCESS,BICARBONATE,BILIRUBIN,BUN,...,iv_total,rate_norepinephrine,rate_epinephrine,rate_phenylephrine,rate_vasopressin,rate_dopamine,vaso_total,PEEP,tidal_volume,plateau_pressure
count,2.214554e+06,2.214554e+06,2.214554e+06,2214554,35887.000000,10392.000000,148225.000000,200877.000000,64517.000000,201879.000000,...,2.214554e+06,179550.000000,15848.000000,80372.000000,29257.000000,9031.000000,269029.000000,6.453870e+05,641031.000000,2.513470e+05
mean,3.496541e+07,1.502912e+07,2.499906e+07,2153-10-22 06:14:26.410846208,3.105270,5.611140,-1.127001,22.785876,2.279678,28.422907,...,3.335892e+02,0.179976,0.120338,1.665180,2.590503,9.474031,2.654010,2.735723e+01,482.765382,8.605728e+02
min,3.000015e+07,1.000003e+07,2.000009e+07,2110-01-11 08:00:00,0.500000,0.000000,-10.000000,2.000000,0.000000,1.000000,...,0.000000e+00,0.000700,0.000864,0.017564,0.016635,0.433513,0.000700,-9.500000e+00,0.000000,0.000000e+00
25%,3.246256e+07,1.252476e+07,2.250096e+07,2133-11-07 08:00:00,2.600000,0.000000,-3.500000,20.000000,0.400000,13.000000,...,0.000000e+00,0.050072,0.020036,0.500229,1.801802,4.002070,0.070197,5.000000e+00,380.500000,1.600000e+01
50%,3.494514e+07,1.506793e+07,2.502963e+07,2153-08-27 00:00:00,3.100000,2.000000,-1.000000,23.000000,0.800000,20.000000,...,5.000000e+01,0.100084,0.040042,1.000167,2.400000,6.008712,0.181608,5.000000e+00,450.000000,2.000000e+01
75%,3.746081e+07,1.753856e+07,2.746327e+07,2173-12-27 12:00:00,3.600000,8.000000,1.000000,25.000000,1.800000,35.000000,...,4.142857e+02,0.200580,0.090034,1.996260,2.408430,10.018067,0.455527,8.500000e+00,520.000000,2.400000e+01
max,3.999986e+07,1.999999e+07,2.999983e+07,2214-07-26 16:00:00,6.800000,86.000000,10.000000,50.000000,78.000000,263.000000,...,1.006000e+05,359.550595,41.142862,880.058706,2400.000000,4000.000000,19992.000000,8.774580e+06,820914.000000,2.111110e+08
std,2.883604e+06,2.892059e+06,2.876734e+06,NaN,0.681144,8.237683,3.824188,4.918844,4.937398,23.908617,...,6.274680e+02,1.280717,0.540329,8.346304,15.399944,57.466350,42.882095,1.186126e+04,2337.715984,4.210891e+05


In [27]:
sampled_withoutlab_withventparams_corrected.MechVent.value_counts()

,count
MechVent,
0,1548811
1,665743


In [ ]:
sampled_withoutlab_withventparams = create_sampled_withoutlab_withventparams_fast(path)
display(sampled_withoutlab_withventparams.head())

Loading overalltable_withoutlab_withventparams...
Performing vectorized 4-hour resampling...
Saved samples withoutlab_withventparams to /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/patient_data


,stay_id,subject_id,hadm_id,start_time,ALBUMIN,BANDS,BASE_EXCESS,BICARBONATE,BILIRUBIN,BUN,...,rate_norepinephrine,rate_epinephrine,rate_phenylephrine,rate_vasopressin,rate_dopamine,vaso_total,PEEP,tidal_volume,plateau_pressure,MechVent
0,30000153,12466550.0,23998182.0,2174-09-29 12:00:00,NaN,NaN,-3.0,19.0,NaN,22.0,...,NaN,NaN,NaN,NaN,NaN,NaN,5.0,537.5,16.0,NaN
1,30000153,12466550.0,23998182.0,2174-09-29 16:00:00,NaN,NaN,-4.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,5.0,626.0,NaN,NaN
2,30000153,12466550.0,23998182.0,2174-09-29 20:00:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,30000153,12466550.0,23998182.0,2174-09-30 00:00:00,NaN,NaN,NaN,23.0,NaN,22.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,30000153,12466550.0,23998182.0,2174-09-30 04:00:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
sampled_withoutlab_withventparams.describe()

,stay_id,subject_id,hadm_id,start_time,ALBUMIN,BANDS,BASE_EXCESS,BICARBONATE,BILIRUBIN,BUN,...,rate_norepinephrine,rate_epinephrine,rate_phenylephrine,rate_vasopressin,rate_dopamine,vaso_total,PEEP,tidal_volume,plateau_pressure,MechVent
count,2.214554e+06,2.214554e+06,2.214554e+06,2214554,35887.000000,10392.000000,148225.000000,200877.000000,64517.000000,201879.000000,...,179550.000000,15848.000000,80372.000000,29257.000000,9031.000000,269029.000000,6.453870e+05,641031.000000,2.513470e+05,0.0
mean,3.496541e+07,1.502912e+07,2.499906e+07,2153-10-22 06:14:26.410846208,3.105270,5.611140,-1.127001,22.785876,2.279678,28.422907,...,0.179976,0.120338,1.665180,2.590503,9.474031,2.654010,2.735723e+01,482.765382,8.605728e+02,NaN
min,3.000015e+07,1.000003e+07,2.000009e+07,2110-01-11 08:00:00,0.500000,0.000000,-10.000000,2.000000,0.000000,1.000000,...,0.000700,0.000864,0.017564,0.016635,0.433513,0.000700,-9.500000e+00,0.000000,0.000000e+00,NaN
25%,3.246256e+07,1.252476e+07,2.250096e+07,2133-11-07 08:00:00,2.600000,0.000000,-3.500000,20.000000,0.400000,13.000000,...,0.050072,0.020036,0.500229,1.801802,4.002070,0.070197,5.000000e+00,380.500000,1.600000e+01,NaN
50%,3.494514e+07,1.506793e+07,2.502963e+07,2153-08-27 00:00:00,3.100000,2.000000,-1.000000,23.000000,0.800000,20.000000,...,0.100084,0.040042,1.000167,2.400000,6.008712,0.181608,5.000000e+00,450.000000,2.000000e+01,NaN
75%,3.746081e+07,1.753856e+07,2.746327e+07,2173-12-27 12:00:00,3.600000,8.000000,1.000000,25.000000,1.800000,35.000000,...,0.200580,0.090034,1.996260,2.408430,10.018067,0.455527,8.500000e+00,520.000000,2.400000e+01,NaN
max,3.999986e+07,1.999999e+07,2.999983e+07,2214-07-26 16:00:00,6.800000,86.000000,10.000000,50.000000,78.000000,263.000000,...,359.550595,41.142862,880.058706,2400.000000,4000.000000,19992.000000,8.774580e+06,820914.000000,2.111110e+08,NaN
std,2.883604e+06,2.892059e+06,2.876734e+06,NaN,0.681144,8.237683,3.824188,4.918844,4.937398,23.908617,...,1.280717,0.540329,8.346304,15.399944,57.466350,42.882095,1.186126e+04,2337.715984,4.210891e+05,NaN


In [28]:
import pandas as pd
import numpy as np

def create_sampled_all_withventparams_fast(path):
    """
    Optimized and fully vectorized pandas equivalent of the SQL for sampled_all_withventparams.
    Produces 4-hour time-binned data from merged lab/non-lab datasets.
    """

    print("Loading and concatenating input datasets...")
    print("Loading overalltable_withoutlab_withventparams...")
    df1 = pd.read_csv(f"{path}/sampled_lab_withventparams.csv",
                     parse_dates=["start_time"])
    # df1 = df1.rename(columns={
    #     'heartrate': 'HeartRate',
    #     'sysbp': 'SysBP',
    #     'diasbp': 'DiasBP',
    #     'meanbp': 'MeanBP',
    #     'resprate': 'RespRate',
    #     'tempc': 'TempC',
    #     'spo2': 'SpO2'
    # })
    df2 = pd.read_csv(f"{path}/sampled_withoutlab_withventparams.csv",
                     parse_dates=["start_time"])
    # Rename time column to match SQL
    # df2 = df2.rename(columns={"charttime":"start_time"})

    df = pd.concat([df1, df2], ignore_index=True)
    print("Merged Data")
    # df = pd.concat([df1, df2], ignore_index=True)

    # Rename icustay_id → stay_id for consistency and easier grouping
    if "icustay_id" in df.columns:
        df.rename(columns={"icustay_id": "stay_id"}, inplace=True)

    id_cols = ["stay_id", "subject_id", "hadm_id"]

    # Sort and set datetime index for resampling
    df = df.sort_values(id_cols + ["start_time"]).set_index("start_time")

    print(f"Data loaded: {len(df):,} rows across {df['stay_id'].nunique():,} stays")

    # Define columns for different aggregation modes
    mean_cols = [
        'ALBUMIN',	'BANDS',	'BASE_EXCESS',	'BICARBONATE',	'BILIRUBIN',	'BUN',	'CALCIUM',	'CARBONDIOXIDE',
        'CHLORIDE',	'CREATININE',	'cum_fluid_balance',	'DiasBP',	'gcs',	'GLUCOSE',	'HeartRate',	'HEMOGLOBIN',
        'INR',	'IonizedCalcium',	'LACTATE',	'MAGNESIUM',	'MeanBP',	'PACO2',	'PAO2',	'PAO2FiO2ratio',
        'PH',	'PLATELET',	'POTASSIUM',	'PT',	'PTT',	'RespRate',	'SGOT',	'SGPT',	'shockindex',	'SODIUM',
        'SpO2',	'SysBP',	'TempC',	'WBC', 'FiO2'
    ]
    sum_cols = ["urineoutput", "iv_total"]
    max_cols = [
         "rate_norepinephrine","rate_epinephrine","rate_phenylephrine",
        "rate_vasopressin","rate_dopamine","vaso_total","PEEP","tidal_volume","plateau_pressure"
    ]

    agg_map = {c: "mean" for c in mean_cols if c in df.columns}
    agg_map.update({c: "sum" for c in sum_cols if c in df.columns})
    agg_map.update({c: "max" for c in max_cols if c in df.columns})
    if "FiO2" in df.columns:
        agg_map["FiO2"] = "mean"
    if "MechVent" in df.columns:
        agg_map["MechVent"] = "mean"

    print("Performing efficient groupby-resample (4-hour bins)...")
    grouped = (
        df.groupby(id_cols)
          .resample("4h")      # time binning within each ICU stay
          .agg(agg_map)
          .reset_index()       # bring back IDs and times
          .rename(columns={"start_time": "start_time"})
    )

    # Apply post-processing rules (as per SQL)
    if "MechVent" in grouped:
        grouped["MechVent"] = (grouped["MechVent"] > 0).astype(int)
    if "gcs" in grouped:
        grouped["gcs"] = grouped["gcs"].round(0)
    if "SysBP" in grouped and "HeartRate" in grouped:
        grouped["shockindex"] = grouped["SysBP"] / grouped["HeartRate"].replace(0, np.nan)
    if "PAO2" in grouped and "FiO2" in grouped:
        grouped["PAO2FiO2ratio"] = grouped["PAO2"] / grouped["FiO2"].replace(0, np.nan) * 100

    # Clean and order
    grouped = grouped.sort_values(id_cols + ["start_time"]).reset_index(drop=True)

    grouped.to_csv(f"{path}/sampled_all_withventparams.csv", index=False)
    print("Saved sampled_all_withventparams to", path)
    return grouped


In [29]:
sampled_all_withventparams = create_sampled_all_withventparams_fast(path)
display(sampled_all_withventparams.head(2))
print(sampled_all_withventparams.describe())

Loading and concatenating input datasets...
Loading overalltable_withoutlab_withventparams...
Merged Data
Data loaded: 2,773,489 rows across 94,458 stays
Performing efficient groupby-resample (4-hour bins)...
Saved sampled_all_withventparams to /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/patient_data


,stay_id,subject_id,hadm_id,start_time,ALBUMIN,BANDS,BASE_EXCESS,BICARBONATE,BILIRUBIN,BUN,...,rate_norepinephrine,rate_epinephrine,rate_phenylephrine,rate_vasopressin,rate_dopamine,vaso_total,PEEP,tidal_volume,plateau_pressure,MechVent
0,30000153,12466550.0,23998182.0,2174-09-29 12:00:00,NaN,NaN,-3.166667,19.0,NaN,22.0,...,NaN,NaN,NaN,NaN,NaN,NaN,5.0,537.5,16.0,1
1,30000153,12466550.0,23998182.0,2174-09-29 16:00:00,NaN,NaN,-4.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,5.0,626.0,NaN,1


            stay_id    subject_id       hadm_id  \
count  2.248324e+06  2.248324e+06  2.248324e+06   
mean   3.496333e+07  1.502315e+07  2.500305e+07   
min    3.000015e+07  1.000003e+07  2.000009e+07   
25%    3.246256e+07  1.251797e+07  2.250184e+07   
50%    3.493873e+07  1.506143e+07  2.503390e+07   
75%    3.745966e+07  1.753377e+07  2.746860e+07   
max    3.999986e+07  1.999999e+07  2.999983e+07   
std    2.882858e+06  2.892779e+06  2.879860e+06   

                          start_time       ALBUMIN         BANDS  \
count                        2248324  45690.000000  13506.000000   
mean   2153-10-04 13:18:35.237130240      3.084401      5.594344   
min              2110-01-11 08:00:00      0.500000      0.000000   
25%              2133-10-02 04:00:00      2.600000      0.000000   
50%              2153-08-10 20:00:00      3.100000      2.000000   
75%              2173-12-23 16:00:00      3.600000      7.000000   
max              2214-07-26 16:00:00      6.800000     86.000000

In [19]:
sampled_all_withventparams.shape

(2248324, 55)

In [30]:
mimic_path = "/content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0"

import pandas as pd

def merge_sampled_with_scdem_withventparams(path, mimic_path):
    """
    Merges sampled data with SIRS/SOFA scores, demographics, and weight per ICU stay as in your SQL.
    Parameters:
        sampled_all_path : CSV with columns including start_time, icustay_id, subject_id, hadm_id, etc.
        sirs_path        : CSV with icustay_id, start_time, sirs
        sofa_path        : CSV with icustay_id, start_time, sofa
        demographics_path: CSV with icustay_id, first_admit_age, gender, icu_readm, elixhauser_score, hospmort90day, dischtime, deathtime, etc.
        weight_path      : CSV with icustay_id, weight
        icustays_path    : CSV with icustay_id, subject_id, hadm_id
        output_path      : CSV path for final merged file

    Returns:
        pd.DataFrame
    """
    print("Loading tables...")
    samp = pd.read_csv(f"{path}/sampled_all_withventparams.csv", parse_dates=['start_time'])
    sr = pd.read_csv(f"{path}/SIRS_sampled_withventparams.csv", parse_dates=['start_time'])
    sf = pd.read_csv(f"{path}/sofa_scores.csv", parse_dates=['start_time'])
    dem = pd.read_csv(f"{path}/demographics2.csv")
    weig = pd.read_csv(f"{path}/weight.csv")
    icu = pd.read_csv(f"{mimic_path}/icu/icustays.csv.gz",
                     usecols=['subject_id', 'hadm_id', 'stay_id'])

    # LEFT JOIN sampled <- getsirs (on icustay_id, start_time)
    merged = samp.merge(sr[['stay_id','start_time','SIRS']],
                        on=['stay_id','start_time'], how='left')
    # LEFT JOIN sampled <- getsofa (on icustay_id, start_time)
    merged = merged.merge(sf[['stay_id','start_time','SOFA']],
                          on=['stay_id','start_time'], how='left')
    # LEFT JOIN sampled <- demographics2 (on icustay_id)
    merged = merged.merge(dem, on='stay_id', how='left', suffixes=('', '_dem'))
    # LEFT JOIN sampled <- getweight2 (on icustay_id)
    merged = merged.merge(weig[['stay_id','weight']], on='stay_id', how='left')
    # INNER JOIN with icustays to bring in subject_id/hadm_id if any missing
    merged = merged.merge(icu[['stay_id','subject_id','hadm_id']],
                          on='stay_id', how='inner', suffixes=('', '_icu'))

    # Set output columns in the required order
    outcols = [
        'stay_id', 'subject_id', 'hadm_id', 'start_time', 'first_admit_age',
        'gender', 'weight', 'ICU_readm', 'elixhauser_score', 'SOFA', 'SIRS',
        'gcs', 'HeartRate', 'SysBP', 'DiasBP', 'MeanBP', 'shockindex', 'RespRate',
        'TempC', 'SpO2', 'POTASSIUM', 'SODIUM', 'CHLORIDE', 'GLUCOSE', 'BUN',
        'CREATININE', 'MAGNESIUM', 'CALCIUM', 'ionizedcalcium', 'CARBONDIOXIDE',
        'SGOT', 'SGPT', 'BILIRUBIN', 'ALBUMIN', 'HEMOGLOBIN', 'WBC', 'PLATELET',
        'PTT', 'PT', 'INR', 'PH', 'PAO2', 'PACO2', 'BASE_EXCESS', 'BICARBONATE',
        'LACTATE', 'PAO2FiO2ratio', 'MechVent', 'FiO2', 'urineoutput',
        'vaso_total', 'iv_total', 'cum_fluid_balance', 'PEEP', 'tidal_volume',
        'plateau_pressure', 'HospMort90day', 'dischtime', 'deathtime'
    ]
    # Add only columns present to avoid key errors
    final_cols = [col for col in outcols if col in merged.columns]
    out = merged[final_cols].sort_values(['stay_id', 'subject_id', 'hadm_id', 'start_time']).reset_index(drop=True)

    out.to_csv(f"{path}/sampled_with_scdem_withventparams.csv", index=False)
    print("Saved sampled_all_withventparams to", path)
    return out


In [31]:
sampled_with_scdem_withventparams = merge_sampled_with_scdem_withventparams(path, mimic_path)
display(sampled_with_scdem_withventparams.head())

Loading tables...
Saved sampled_all_withventparams to /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/patient_data


,stay_id,subject_id,hadm_id,start_time,first_admit_age,gender,weight,ICU_readm,elixhauser_score,SOFA,...,urineoutput,vaso_total,iv_total,cum_fluid_balance,PEEP,tidal_volume,plateau_pressure,HospMort90day,dischtime,deathtime
0,30000153,12466550.0,23998182.0,2174-09-29 12:00:00,60.0,M,73.0,1,6.0,4,...,0.0,NaN,1400.0,-840.000000,5.0,537.5,16.0,0,2174-10-15 15:24:00,NaN
1,30000153,12466550.0,23998182.0,2174-09-29 16:00:00,60.0,M,73.0,1,6.0,7,...,0.0,NaN,50.0,-1178.333333,5.0,626.0,NaN,0,2174-10-15 15:24:00,NaN
2,30000153,12466550.0,23998182.0,2174-09-29 20:00:00,60.0,M,73.0,1,6.0,6,...,0.0,NaN,400.0,-1320.000000,NaN,NaN,NaN,0,2174-10-15 15:24:00,NaN
3,30000153,12466550.0,23998182.0,2174-09-30 00:00:00,60.0,M,73.0,1,6.0,5,...,0.0,NaN,0.0,-1438.333333,NaN,NaN,NaN,0,2174-10-15 15:24:00,NaN
4,30000153,12466550.0,23998182.0,2174-09-30 04:00:00,60.0,M,73.0,1,6.0,5,...,0.0,NaN,1000.0,-1636.666667,NaN,NaN,NaN,0,2174-10-15 15:24:00,NaN


In [31]:
sampled_with_scdem_withventparams.shape

(2248324, 57)

In [5]:
samp = pd.read_csv(f"{path}/sampled_all_withventparams.csv", parse_dates=['start_time'])

In [11]:
df2.MechVent.value_counts()

,count
MechVent,


In [7]:
df1 = pd.read_csv(f"{path}/sampled_lab_withventparams.csv",
                     parse_dates=["start_time"])
df2 = pd.read_csv(f"{path}/sampled_withoutlab_withventparams.csv",
                     parse_dates=["start_time"])